In [ ]:
# Import libraries
import os
import re
import json
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

# Set output directory
outputs_dir = 'outputs'  # relative to transformer_notebook

# Regex to extract hyperparameters from folder name (e.g., exp10_d896_b1)
exp_pattern = re.compile(r'd(\d+)_b(\d+)(?:_e(\d+))?')
lr_pattern = re.compile(r'lr([\deE.-]+)')

# Scan all experiment folders
experiments = []
for folder in os.listdir(outputs_dir):
    exp_path = os.path.join(outputs_dir, folder)
    hist_path = os.path.join(exp_path, 'training_history.json')
    if os.path.isdir(exp_path) and os.path.exists(hist_path):
        # Extract d_model, batch_size, epochs, lr from folder name
        m = exp_pattern.search(folder)
        d_model = int(m.group(1)) if m else None
        batch_size = int(m.group(2)) if m else None
        epochs = int(m.group(3)) if m and m.group(3) else None
        lr_match = lr_pattern.search(folder)
        lr = float(lr_match.group(1)) if lr_match else None
        # Load training_history.json
        with open(hist_path) as f:
            hist = json.load(f)
        best_per = hist.get('best_per', None)
        # Try to get lr from config if not in folder name
        if lr is None and 'learning_rates' in hist and hist['learning_rates']:
            lr = max(hist['learning_rates'])
        experiments.append({
            'folder': folder,
            'd_model': d_model,
            'batch_size': batch_size,
            'epochs': epochs,
            'lr': lr,
            'best_per': best_per
        })

# Create DataFrame
exp_df = pd.DataFrame(experiments)
exp_df = exp_df.dropna(subset=['d_model', 'batch_size', 'lr', 'best_per'])
exp_df.sort_values('best_per', inplace=True)
exp_df.reset_index(drop=True, inplace=True)

# Show summary table
exp_df.head(10)

## 3D Scatter Plot: PER vs d_model, batch_size, lr

This plot shows how the best PER varies with model dimension, batch size, and learning rate. Lower PER is better.

In [ ]:
# 3D scatter plot
fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(exp_df['d_model'], exp_df['batch_size'], exp_df['lr'], c=exp_df['best_per'], cmap='viridis', s=80)
ax.set_xlabel('d_model')
ax.set_ylabel('batch_size')
ax.set_zlabel('learning rate')
plt.title('Best PER by d_model, batch_size, lr')
cbar = plt.colorbar(sc, ax=ax, pad=0.1)
cbar.set_label('Best PER (lower is better)')
plt.show()

## Pairwise 2D Plots: PER vs Each Hyperparameter

These plots help you see which hyperparameter has the strongest effect on PER.

In [ ]:
# Pairwise 2D scatter plots
fig, axes = plt.subplots(1, 3, figsize=(18,5))
sns.scatterplot(x='d_model', y='best_per', data=exp_df, ax=axes[0], hue='batch_size', palette='viridis', s=80)
axes[0].set_title('PER vs d_model')
axes[0].set_ylabel('Best PER')
axes[0].set_xlabel('d_model')
sns.scatterplot(x='batch_size', y='best_per', data=exp_df, ax=axes[1], hue='d_model', palette='viridis', s=80)
axes[1].set_title('PER vs batch_size')
axes[1].set_ylabel('Best PER')
axes[1].set_xlabel('batch_size')
sns.scatterplot(x='lr', y='best_per', data=exp_df, ax=axes[2], hue='d_model', palette='viridis', s=80)
axes[2].set_title('PER vs learning rate')
axes[2].set_ylabel('Best PER')
axes[2].set_xlabel('learning rate')
plt.tight_layout()
plt.show()

## How to Use These Plots

- **Look for lowest PER points**: These are your best experiments.
- **Trends**: If PER drops as d_model increases, try even larger models. If batch_size or lr shows a sweet spot, focus future runs there.
- **Next steps**: Use these visualizations to pick the next set of experiments, e.g., try intermediate values or combine best settings.